# Terror Infinity RPG — Team Portraits

Generates **2 portraits per character** using **Animagine XL 3.1** (SDXL):
- `{slug}-fullbody.png` — head-to-toe full body, used on the character sheet
- `{slug}-closeup.png` — tight face/shoulder crop, used in the battle card

**Before running:**
1. Runtime → Change runtime type → **T4 GPU** (16 GB VRAM required)
2. Run all cells in order
3. Model downloads ~6.5 GB on first run (~2-3 min, cached for session)

Output is saved to Google Drive `terror-infinity-portraits/` and downloaded as a zip.

In [ ]:
# ── Cell 1: Check GPU ─────────────────────────────────────────────────────────
!nvidia-smi
import torch
print(f"\nCUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU : {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    if torch.cuda.get_device_properties(0).total_memory < 7e9:
        print("⚠  VRAM < 8 GB — SDXL requires at least 8 GB. Switch to T4 runtime.")

In [ ]:
# ── Cell 2: Install dependencies ─────────────────────────────────────────────
!pip install -q -U diffusers transformers accelerate safetensors

In [ ]:
# ── Cell 3: Mount Google Drive ────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
print("Drive mounted ✓  Output → MyDrive/terror-infinity-portraits/")

In [ ]:
# ── Cell 4: Load Animagine XL 3.1 ────────────────────────────────────────────
import torch, os
from diffusers import StableDiffusionXLPipeline, DPMSolverMultistepScheduler

# Reduces memory fragmentation — helps prevent OOM on borderline allocations
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

print("Loading Animagine XL 3.1 ...")
pipe = StableDiffusionXLPipeline.from_pretrained(
    "cagliostrolab/animagine-xl-3.1",
    torch_dtype=torch.float16,
    use_safetensors=True,
)
pipe.scheduler = DPMSolverMultistepScheduler.from_config(
    pipe.scheduler.config, use_karras_sigmas=True,
)
pipe = pipe.to("cuda")
pipe.enable_attention_slicing()
pipe.enable_vae_tiling()   # ← fixes OOM: VAE decode runs in tiles instead of all at once
                            #   the 30/30 steps finish fine but the decode at 832×1216
                            #   needs ~248 MB more than what's free — tiling solves this

used  = torch.cuda.memory_allocated() / 1e9
total = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"\nPipeline ready ✓")
print(f"VRAM used: {used:.1f} GB / {total:.1f} GB")

In [ ]:
# ── Cell 5: Character definitions ─────────────────────────────────────────────
#
# FRAMING prefixes:
#   _QA_FULL  → full body head-to-toe (character sheet)
#   _QA_CLOSE → tight face/shoulder crop (battle card)
#
# NEGATIVES are split — fullbody must NOT block legs/feet:
#   NEG_FULL  → no body blocking, but blocks cropped renders
#   NEG_CLOSE → blocks full body so model crops tight to face
#
# CLIP 77-token limit applies to positive AND negative independently.
# Keep both under 77. Anti-duplicate tags go first (never truncated).

_QA_FULL  = ('masterpiece, best quality, very aesthetic, '
             '2.5D, anime style, full color, '
             'full body, standing, feet visible, solo, dark background, dramatic lighting, ')

_QA_CLOSE = ('masterpiece, best quality, very aesthetic, '
             '2.5D, anime style, full color, '
             'close-up, face focus, solo, dark background, ')

# Trimmed negatives — each comfortably under 77 tokens
NEG_FULL = (
    'duplicate, 2boys, 2girls, multiple people, '
    'bad, worst quality, low quality, blurry, deformed, bad anatomy, '
    'monochrome, grayscale, '
    'child, loli, shota, '
    'cropped, cut off, lower body missing'
)
NEG_CLOSE = NEG_FULL + ', full body, legs, feet'

# ── Character list ─────────────────────────────────────────────────────────────
# (slug, base_appearance, outfit_hint)
CHARACTERS = [
    ('igor-pundzin',
     '1boy, mature male, jet black hair slicked back, brown eyes, medium skin, lean slim build, light stubble',
     'dark jacket, grey t-shirt'),

    ('yuki-tanaka',
     '1girl, adult female, Japanese, black hair long straight, dark brown eyes, fair skin, slender',
     'pale blouse, slight smile'),

    ('marcus-webb',
     '1boy, mature male, brown hair short, green eyes, tan skin, muscular, light beard',
     'dark tactical shirt, cargo pants'),

    ('lena-fischer',
     '1girl, adult female, blonde hair tied back, blue eyes, fair skin, slender',
     'white coat, stethoscope'),

    ('ji-ho-park',
     '1boy, adult male, Korean, black hair short, dark eyes, light skin, lean',
     'grey hoodie'),

    ('amara-osei',
     '1girl, adult female, dark skin, black natural hair, brown eyes, muscular athletic',
     'black sports jacket, tank top'),

    ('anatoly-kravchenko',
     '1boy, middle-aged man, grey hair short, grey eyes, light skin, lean, heavy stubble',
     'dark trench coat'),

    ('isabelle-laurent',
     '1girl, adult female, dark brown hair, hazel eyes, fair skin, slight build',
     'white lab coat'),

    ('kenji-watanabe',
     '1boy, mature male, black hair slicked back, dark eyes, light skin, muscular, stern',
     'black formal suit'),

    ('priya-sharma',
     '1girl, adult female, black hair long, dark eyes, brown skin, lean athletic',
     'university hoodie'),
]

total = len(CHARACTERS) * 2
print(f"Characters : {len(CHARACTERS)}")
print(f"Shots/char : 2  (fullbody + closeup)")
print(f"Total      : {total} images  (~{max(1, total * 35 // 60)} min on Colab T4)")
print()

# ── Token sanity check ───────────────────────────────────────────────────────
try:
    from transformers import CLIPTokenizer
    tok = CLIPTokenizer.from_pretrained("openai/clip-vit-base-patch32")
    print(f"  {'Character':<22} {'F+':>4} {'F-':>4} {'C+':>4} {'C-':>4}  Status")
    print("  " + "-" * 48)
    ok = True
    for slug, base_desc, outfit in CHARACTERS:
        p_full  = _QA_FULL  + base_desc + ', ' + outfit + ', looking at viewer'
        p_close = _QA_CLOSE + base_desc + ', looking at viewer'
        tf  = len(tok(p_full  )["input_ids"])
        tnf = len(tok(NEG_FULL )["input_ids"])
        tc  = len(tok(p_close )["input_ids"])
        tnc = len(tok(NEG_CLOSE)["input_ids"])
        worst = max(tf, tnf, tc, tnc)
        flag = "⚠ OVER" if worst > 77 else "✓"
        if worst > 77: ok = False
        print(f"  {slug:<22} {tf:>4} {tnf:>4} {tc:>4} {tnc:>4}   {flag}")
    print()
    print("All within 77 tokens ✓" if ok else "⚠ Shorten the flagged prompts before generating.")
except Exception as e:
    print(f"(Token check skipped: {e})")

In [ ]:
# ── Cell 6: Generate all portraits ────────────────────────────────────────────
import hashlib, os
from IPython.display import display, Image as IPImage
from PIL import Image as PILImage

OUTPUT_DIR = '/content/portraits'
os.makedirs(OUTPUT_DIR, exist_ok=True)

IMG_W, IMG_H = 832, 1216   # tall portrait — ideal for full body
STEPS        = 30           # extra steps help with full-body anatomy
CFG          = 7.0

def stable_seed(text):
    """Deterministic seed from slug — same seed = same face across both shots."""
    return int(hashlib.md5(text.encode()).hexdigest()[:8], 16) % 999983

results = []

print("=" * 58)
print("  TEAM PORTRAITS  (fullbody + closeup per character)")
print("=" * 58)

for idx, (slug, base_desc, outfit) in enumerate(CHARACTERS):
    seed    = stable_seed(slug)
    p_full  = _QA_FULL  + base_desc + ', ' + outfit + ', looking at viewer'
    p_close = _QA_CLOSE + base_desc + ', looking at viewer'

    print(f"\n[{idx+1}/{len(CHARACTERS)}]  {slug}  (seed {seed})")

    # ── Full body (head to toe) ───────────────────────────────────────────────
    print("  fullbody ... ", end='', flush=True)
    gen = torch.Generator('cuda').manual_seed(seed)
    img_full = pipe(
        prompt=p_full, negative_prompt=NEG_FULL,
        width=IMG_W, height=IMG_H,
        num_inference_steps=STEPS, guidance_scale=CFG,
        generator=gen,
    ).images[0]
    path_full = f'{OUTPUT_DIR}/{slug}-fullbody.png'
    img_full.save(path_full)
    results.append((f'{slug}-fullbody', path_full))
    print('saved')

    # ── Closeup (same seed → same face) ──────────────────────────────────────
    print("  closeup  ... ", end='', flush=True)
    gen = torch.Generator('cuda').manual_seed(seed)  # reset to same seed
    img_close = pipe(
        prompt=p_close, negative_prompt=NEG_CLOSE,
        width=IMG_W, height=IMG_H,
        num_inference_steps=STEPS, guidance_scale=CFG,
        generator=gen,
    ).images[0]
    path_close = f'{OUTPUT_DIR}/{slug}-closeup.png'
    img_close.save(path_close)
    results.append((f'{slug}-closeup', path_close))
    print('saved')

    # ── Side-by-side preview ─────────────────────────────────────────────────
    THUMB_W, THUMB_H = 280, 420
    preview = PILImage.new('RGB', (THUMB_W * 2 + 6, THUMB_H), color=(25, 25, 25))
    preview.paste(img_full.resize( (THUMB_W, THUMB_H)), (0,           0))
    preview.paste(img_close.resize((THUMB_W, THUMB_H)), (THUMB_W + 6, 0))
    preview.save(f'{OUTPUT_DIR}/{slug}-preview.png')
    display(IPImage(f'{OUTPUT_DIR}/{slug}-preview.png', width=580))

print(f"\n✓  {len(results)} portraits generated  ({len(CHARACTERS)} characters × 2 shots)")

In [ ]:
# ── Cell 7: Save to Google Drive + download zip ───────────────────────────────
import shutil, os
from google.colab import files

DRIVE_OUTPUT = '/content/drive/MyDrive/terror-infinity-portraits'
os.makedirs(DRIVE_OUTPUT, exist_ok=True)

copied = 0
for slug, path in results:
    if os.path.exists(path):
        shutil.copy(path, f'{DRIVE_OUTPUT}/{slug}.png')
        copied += 1

print(f'Saved {copied} portraits to Google Drive → terror-infinity-portraits/')

zip_src = '/content/portraits'
shutil.make_archive(zip_src, 'zip', OUTPUT_DIR)
print('Downloading team-portraits.zip ...')
files.download('/content/portraits.zip')